# Инференс DGCA Fusion (RESD) на тестовой выборке

Загружает BERT + WavLM backbone с HuggingFace, DGCA fusion веса из `.pt` файла,  
транскрибирует аудио через Whisper, оценивает на `Aniemore/resd` test.

## 1. Install & clone

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'librosa', 'soundfile', 'scikit-learn', 'tqdm',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done. CWD:', os.getcwd())

## 2. Config

In [ ]:
import warnings, pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoFeatureExtractor,
    AutoProcessor, AutoModelForSpeechSeq2Seq,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report,
)
from tqdm.auto import tqdm
import librosa
warnings.filterwarnings('ignore')

# ── путь к .pt файлу с весами fusion ─────────────────────────────────────────
FUSION_PT  = '/kaggle/input/datasets/aleksandribryanov/dgca-resd/best_dgca_resd.pt'

# ── модели ────────────────────────────────────────────────────────────────────
WHISPER_MODEL = 'artyomboyko/whisper-small-ru-v2'
BERT_MODEL    = 'Aniemore/rubert-tiny2-russian-emotion-detection'
WAVLM_MODEL   = 'Aniemore/wavlm-emotion-russian-resd'

FUSION_DIM  = 1024
FUSION_DROP = 0.0    # dropout=0 при инференсе (model.eval() уже отключает, но явно)
NUM_HEADS   = 4
BATCH_SIZE  = 8
SR_TARGET   = 16_000
MAX_TEXT_LEN = 128
MAX_AUDIO_S  = 10.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_LABELS = ['happiness', 'sadness', 'anger', 'fear', 'disgust', 'enthusiasm', 'neutral']
NUM_CLASSES = len(RESD_LABELS)

## 3. Загрузка backbone моделей (замороженные)

In [ ]:
print(f'Loading BERT: {BERT_MODEL}')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_backbone  = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_backbone.eval()
BERT_DIM = bert_backbone.config.hidden_size
print(f'  hidden_size: {BERT_DIM}')

print(f'Loading WavLM: {WAVLM_MODEL}')
wavlm_processor = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm_backbone  = AutoModel.from_pretrained(WAVLM_MODEL).to(device)
wavlm_backbone.eval()
WAVLM_DIM = wavlm_backbone.config.hidden_size
print(f'  hidden_size: {WAVLM_DIM}')

## 4. DGCA Fusion модуль + загрузка весов

In [ ]:
class DGCAFusion(nn.Module):
    def __init__(self, d_text, d_audio, D=512, num_heads=4, num_classes=7, dropout=0.0):
        super().__init__()
        self.proj_text  = nn.Linear(d_text,  D)
        self.proj_audio = nn.Linear(d_audio, D)
        self.mha_t2a    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.mha_a2t    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.ln_text    = nn.LayerNorm(D)
        self.ln_audio   = nn.LayerNorm(D)
        self.gate_text  = nn.Linear(D, D)
        self.gate_audio = nn.Linear(D, D)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(D, num_classes)

    def forward(self, h_text, h_audio):
        T = self.proj_text(h_text).unsqueeze(1)
        A = self.proj_audio(h_audio).unsqueeze(1)
        T_ref = self.ln_text(T  + self.mha_t2a(T, A, A)[0]).squeeze(1)
        A_ref = self.ln_audio(A + self.mha_a2t(A, T, T)[0]).squeeze(1)
        gates = F.softmax(torch.stack([self.gate_text(T_ref), self.gate_audio(A_ref)], dim=-1), dim=-1)
        fused = gates[..., 0] * T_ref + gates[..., 1] * A_ref
        return self.classifier(self.dropout(fused))


print(f'Loading fusion weights from {FUSION_PT}')
ckpt = torch.load(FUSION_PT, map_location=device, weights_only=False)
print(f'  saved at step={ckpt["step"]}  val_wacc={ckpt["val_wacc"]:.4f}')

# определяем FUSION_DIM из самого чекпоинта
fusion_dim_ckpt = ckpt['fusion']['proj_text.weight'].shape[0]
print(f'  FUSION_DIM from checkpoint: {fusion_dim_ckpt}')

fusion = DGCAFusion(
    d_text=BERT_DIM, d_audio=WAVLM_DIM,
    D=fusion_dim_ckpt, num_heads=NUM_HEADS, num_classes=NUM_CLASSES,
    dropout=0.0,
).to(device)
fusion.load_state_dict(ckpt['fusion'])
fusion.eval()
print(f'Fusion loaded. Params: {sum(p.numel() for p in fusion.parameters()):,}')

## 5. Транскрипция RESD test через Whisper

In [ ]:
print(f'Loading Whisper: {WHISPER_MODEL}')
asr_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
asr_model     = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
asr_model.eval()

@torch.no_grad()
def transcribe(wav_np):
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
    inputs = asr_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
    ids    = asr_model.generate(
        inputs.input_features.to(device, dtype=dtype),
        language='ru', task='transcribe',
    )
    return asr_processor.batch_decode(ids, skip_special_tokens=True)[0]


print('Loading Aniemore/resd test split...')
ds_test = load_dataset('Aniemore/resd', split='test')
print(f'Test samples: {len(ds_test)}')

test_recs = []
for ex in tqdm(ds_test, desc='Transcribing'):
    wav = np.array(ex['speech']['array'], dtype=np.float32)
    sr  = ex['speech']['sampling_rate']
    if sr != SR_TARGET:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
    text = transcribe(wav)
    test_recs.append({
        'text':  text,
        'audio': wav,
        'label': RESD_LABEL2ID[ex['emotion']],
    })

del asr_model, asr_processor
torch.cuda.empty_cache()
print(f'Transcription done. Example: "{test_recs[0]["text"]}" → {RESD_LABELS[test_recs[0]["label"]]}')

## 6. Инференс

In [ ]:
MAX_AUDIO_LEN = int(MAX_AUDIO_S * SR_TARGET)


@torch.no_grad()
def encode_text(input_ids, attention_mask):
    return bert_backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]


@torch.no_grad()
def encode_audio(audio, audio_mask):
    embeddings = []
    for i in range(audio.shape[0]):
        wav_np = audio[i][audio_mask[i].bool()].cpu().numpy()
        inputs = wavlm_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
        hidden = wavlm_backbone(inputs['input_values'].to(device)).last_hidden_state
        embeddings.append(hidden.mean(dim=1).squeeze(0))
    return torch.stack(embeddings)


def collate_fn(batch):
    max_len = max(b['audio'].shape[0] for b in batch)
    audios  = torch.zeros(len(batch), max_len)
    masks   = torch.zeros(len(batch), max_len)
    for i, b in enumerate(batch):
        L = b['audio'].shape[0]
        audios[i, :L] = b['audio']; masks[i, :L] = 1.0
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'audio': audios, 'audio_mask': masks,
        'label': torch.stack([b['label'] for b in batch]),
    }


class InferDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r   = self.records[i]
        enc = bert_tokenizer(
            r['text'], truncation=True, padding='max_length',
            max_length=MAX_TEXT_LEN, return_tensors='pt')
        wav = r['audio'][:MAX_AUDIO_LEN]
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'audio':          torch.tensor(wav, dtype=torch.float32),
            'label':          torch.tensor(r['label'], dtype=torch.long),
        }


loader = DataLoader(InferDataset(test_recs), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2, collate_fn=collate_fn)

preds_all, labels_all = [], []
for batch in tqdm(loader, desc='Inference'):
    ids   = batch['input_ids'].to(device)
    mask  = batch['attention_mask'].to(device)
    audio = batch['audio'].to(device)
    amask = batch['audio_mask'].to(device)
    with torch.no_grad():
        logits = fusion(encode_text(ids, mask), encode_audio(audio, amask))
    preds_all.append(logits.argmax(-1).cpu().numpy())
    labels_all.append(batch['label'].numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)
print(f'Inference done: {len(preds)} samples')

## 7. Результаты

In [ ]:
print('=== DGCA Fusion (BERT + WavLM) on RESD test ===')
print(f'Accuracy          : {accuracy_score(labels, preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(labels, preds):.4f}')
print(f'F1 Macro          : {f1_score(labels, preds, average="macro",    zero_division=0):.4f}')
print(f'F1 Weighted       : {f1_score(labels, preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(labels, preds, target_names=RESD_LABELS, zero_division=0))